# Data Pre-processing

In [ ]:
import sklearn as sk
import pandas as pd

## Import Data

In [ ]:
df = pd.read_csv("US_Accidents_March23.csv")
len(df)

## Basic Data Preprocessing
To simplify your project a bit, use only the data instances that contain at most 1 missing value. According to our counts:
There are 3,554,549 data instances with no missing values:
There are 244,702 data instances with exactly one missing value.
Hence, your dataset for the project should contain 3,554,549 + 244,702 = 3,799,251

In [ ]:
df.dropna(thresh=len(df.columns) - 1, inplace=True)
len(df)

In [ ]:
df.drop(columns=['ID', 'Description', 'Country', 'Source', 'Turning_Loop'], inplace = True)
print(df.columns)

In [ ]:
basic_df = df.copy(deep = True)

# Handle datetime -> integer by extracting integer values
basic_df["Start_Time"] = pd.to_datetime(basic_df["Start_Time"], format="ISO8601")
basic_df["End_Time"] = pd.to_datetime(basic_df["End_Time"], format="ISO8601")

basic_df["Duration_Minutes"] = (
    basic_df["End_Time"] - basic_df["Start_Time"]
).dt.total_seconds() / 60.0
basic_df["Start_Hour"] = basic_df["Start_Time"].dt.hour
basic_df["Start_DayOfWeek"] = basic_df["Start_Time"].dt.dayofweek  # 0 = Monday, 6 = Sunday
basic_df["Start_Month"] = basic_df["Start_Time"].dt.month
basic_df.drop(columns=["Start_Time", "End_Time"], inplace=True)

In [ ]:
# Drop location cols other than lat/long
basic_df.drop(
    columns=[
        "City", "County", "Zipcode", "Airport_Code", "State"
    ],
    inplace=True,
)

In [ ]:
#Convert sunset measures
sunset_fields = ["Sunrise_Sunset", "Civil_Twilight", "Nautical_Twilight", "Astronomical_Twilight"]
for f in sunset_fields:
    basic_df[f] = basic_df[f].map({"Day": 1, "Night": 0}) # day = 1, night = 0

In [ ]:
# One-hot encode wind direction and weather condition as dense columns
weather_columns = ["Wind_Direction", "Weather_Condition"]
basic_df = pd.get_dummies(
    basic_df,
    columns=weather_columns,
    dummy_na=True,
)

In [ ]:
#Drop timezone, weather
basic_df.drop(columns = ["Timezone", "Weather_Timestamp"], inplace = True)

In [ ]:
# Fill missing numeric values one column at a time to avoid allocating another full-size frame.
for column in basic_df.columns:
    if (column == "Street" or column == "Weather_Condition" or column == "Wind_Direction"):
        continue
    
    missing = basic_df[column].isna()
    if missing.any():
        fill_value = basic_df[column].median()
        if pd.isna(fill_value):
            fill_value = 0
        basic_df[column] = basic_df[column].fillna(fill_value)

In [ ]:
basic_df.info()

In [ ]:
basic_df.head()

## Advanced Preprocessing

In [ ]:
import numpy as np


def add_road_type(f):
    hwy_re = (
        r'\b[A-Z]{1,2}-\d+'
        r'|\b(?:Hwy|Fwy|Expy|Pkwy|Tpke|Turnpike|Highway|Freeway|Expressway|Interstate|Route|Rte|SR)\b'
    )
    local_re = (
        r'\b(?:St|Street|Rd|Road|Ave|Avenue|Blvd|Boulevard|Ln|Lane|Dr|Drive|'
        r'Ct|Court|Pl|Place|Way|Pike|Cir|Circle|Trl|Trail|Ter|Terrace)\b'
    )

    street = f['Street']
    f['road_type'] = np.select(
        [
            street.isna(),
            street.str.contains(hwy_re, regex=True, na=False),
            street.str.contains(local_re, regex=True, na=False),
        ],
        [None, 'highway', 'local_street'],
        default='other',
    )
    return f


def weather_type(cond):
    if pd.isna(cond):
        return np.nan
    if 'Fair' in cond or cond == 'Clear':
        return 'clear'
    if 'Thunder' in cond or 'T-Storm' in cond:
        return 'storm'
    if any(term in cond for term in ('Rain', 'Drizzle', 'Precipitation', 'Showers')):
        return 'rain'
    if cond == 'Wintry Mix' or 'Snow' in cond:
        return 'snow'
    if 'Cloudy' in cond or cond == 'Cloudy' or 'Clouds' in cond or 'Overcast' in cond:
        return 'cloudy'
    if any(term in cond for term in ('Fog', 'Mist', 'Smoke', 'Haze', 'Dust', 'Sand')):
        return 'low-visibility'
    if any(term in cond for term in ('Sleet', 'Hail', 'Ice Pellets', 'Tornado')):
        return 'storm'
    if 'Windy' in cond or 'Squall' in cond:
        return 'windy'
    return 'cloudy'


def add_rush_hour(hour):
    return int((7 <= hour <= 9) or (16 <= hour <= 19))


def add_wind_direction_features(F):
    direction_degrees = {
        'N': 0.0,
        'NNE': 22.5,
        'NE': 45.0,
        'ENE': 67.5,
        'E': 90.0,
        'ESE': 112.5,
        'SE': 135.0,
        'SSE': 157.5,
        'S': 180.0,
        'SSW': 202.5,
        'SW': 225.0,
        'WSW': 247.5,
        'W': 270.0,
        'WNW': 292.5,
        'NW': 315.0,
        'NNW': 337.5,
        'NORTH': 0.0,
        'EAST': 90.0,
        'SOUTH': 180.0,
        'WEST': 270.0,
    }

    direction = F['Wind_Direction'].astype('string').str.strip().str.upper()
    degrees = direction.map(direction_degrees).astype('float64')
    radians = np.deg2rad(degrees)
    calm = direction.eq('CALM').fillna(False)
    variable = direction.isin(['VAR', 'VARIABLE', 'VRB']).fillna(False)

    F['Wind_Direction_sin'] = np.sin(radians).fillna(0.0)
    F['Wind_Direction_cos'] = np.cos(radians).fillna(0.0)
    F['Wind_Direction_calm'] = calm.astype('boolean')
    F['Wind_Direction_variable'] = variable.astype('boolean')

    return F.drop(columns=['Wind_Direction'])

### Grouped Features

In [ ]:
def add_grouped_features(F):
    poi_type_groups = {
        'traffic_controlled': ['Traffic_Signal', 'Stop', 'Give_Way'],
        'junctions': ['Junction', 'Roundabout', 'No_Exit'],
        'traffic_calming': ['Bump', 'Traffic_Calming'],
        'pedestrians': ['Crossing', 'Station', 'Amenity'],
    }

    for name, cols in poi_type_groups.items():
        F[name] = F[cols].fillna(False).astype(bool).any(axis=1).astype(int)
    F = F.drop(columns=[col for cols in poi_type_groups.values() for col in cols])

    F = add_road_type(F)
    F = F.drop(columns=['Street'])
    F['tod_lighting'] = np.select(
        [F['Sunrise_Sunset'] == 'Day', F['Civil_Twilight'] == 'Day'],
        ['day', 'twilight'],
        default='night',
    )
    return F.drop(
        columns=['Sunrise_Sunset', 'Civil_Twilight', 'Nautical_Twilight', 'Astronomical_Twilight']
    )

### Weather and Time Features

In [ ]:
def add_weather_and_time_features(F):
    F['Weather_Condition'] = F['Weather_Condition'].apply(weather_type)

    F['Start_Time'] = pd.to_datetime(F['Start_Time'], format='ISO8601')
    F['End_Time'] = pd.to_datetime(F['End_Time'], format='ISO8601')
    F['Duration_Minutes'] = (F['End_Time'] - F['Start_Time']).dt.total_seconds() / 60.0
    F['Start_Hour'] = F['Start_Time'].dt.hour
    F['rush_hour'] = F['Start_Hour'].apply(add_rush_hour)
    F['Start_DayOfWeek'] = F['Start_Time'].dt.dayofweek
    F['Start_Month'] = F['Start_Time'].dt.month
    return F.drop(columns=['Start_Time', 'End_Time'])

### Numeric Encoding and Cleanup

In [ ]:
def encode_and_clean_advanced(F):
    F = F.drop(
        columns=[
            'City', 'County', 'Zipcode', 'Airport_Code', 'State',
            'End_Lat', 'End_Lng', 'Timezone', 'Weather_Timestamp',
        ]
    )
    F = add_wind_direction_features(F)

    category_codes = {
        'Weather_Condition': {
            'clear': 0,
            'cloudy': 1,
            'rain': 2,
            'snow': 3,
            'storm': 4,
            'low-visibility': 5,
            'windy': 6,
        },
        'road_type': {
            'highway': 0,
            'local_street': 1,
            'other': 2,
        },
        'tod_lighting': {
            'day': 0,
            'twilight': 1,
            'night': 2,
        },
    }
    for column, mapping in category_codes.items():
        F[column] = F[column].map(mapping).fillna(-1).astype('int8')

    numeric_columns = F.select_dtypes(include=['number', 'bool']).columns
    for column in numeric_columns:
        missing = F[column].isna()
        if missing.any():
            fill_value = F[column].median()
            if pd.isna(fill_value):
                fill_value = 0
            F[column] = F[column].fillna(fill_value)

    return F

### Build Advanced Dataframe

In [ ]:
def advanced_preprocessing(fold):
    F = fold.copy()
    F = add_grouped_features(F)
    F = add_weather_and_time_features(F)
    return encode_and_clean_advanced(F)


advanced_processed_data = advanced_preprocessing(df)
assert advanced_processed_data.select_dtypes(exclude=['number', 'bool']).empty
advanced_processed_data.info()

### Check dataframe

In [ ]:
advanced_processed_data.info()

In [ ]:
advanced_processed_data.head()

# Export

In [ ]:
%pip install pyarrow

basic_df.to_parquet(
    "basic_processed_data.parquet",
    engine="pyarrow",
    compression="zstd",
    index=True
)


advanced_processed_data.to_parquet(
    "advanced_processed_data.parquet",
    engine="pyarrow",
    compression="zstd",
    index=True
)